# 03 — Transformer Fine-Tuning

**Project**: From Clinical Jargon to Plain Language — Medical Text Simplification  
**Purpose**: Fine-tune T5-small and SciFive on PLABA+Cochrane train split. Both models use identical code — only `MODEL_NAME` changes.  
**Output**: `predictions/t5_small.jsonl`, `predictions/scifive.jsonl`, `results/metrics.csv` (t5_small + scifive rows)  
**Run**: Top-to-bottom on Colab T4 GPU. Restart kernel after install cell, then re-run from Config cell.

## 0. Colab Setup

Mount Drive and clone repo. Run ONCE per session before kernel restart.

In [ ]:
from google.colab import userdata, drive
import os, sys, shutil

# Mount Drive
drive.mount('/drive')

# Clone repo if not already present
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_URL = f'https://{GITHUB_TOKEN}@github.com/IbrahimHanafy2222/NLP-Project.git'
if not os.path.exists('/content/NLP-Project'):
    import subprocess
    result = subprocess.run(['git', 'clone', GITHUB_URL], capture_output=True, text=True, cwd='/content')
    print(result.stdout or result.stderr)

# Set working directory
project_root = '/content/NLP-Project'
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print('Working directory:', os.getcwd())

# Load processed data from Drive if not present
if not os.path.exists('data/processed'):
    shutil.copytree('/drive/MyDrive/NLP_Project/processed', 'data/processed')
    print('Loaded data/processed from Drive')
else:
    print('data/processed already present')

## 1. Install

Installs on first run only. Kernel restarts automatically — re-run from Config cell after restart.

In [ ]:
import importlib.util as _ilu
if _ilu.find_spec('textstat') is None:
    get_ipython().system('pip install git+https://github.com/feralvam/easse.git sacrebleu "transformers>=4.35" datasets==2.18.0 torch spacy==3.7.4 pandas==2.2.1 textstat scikit-learn==1.4.1.post1')
    get_ipython().system('python -m spacy download en_core_web_sm')
    print('Packages installed. Runtime restarting — re-run from Config cell after restart.')
    import os; os.kill(os.getpid(), 9)
else:
    print('Packages already installed. Continuing.')

## 2. Config

All hyperparameters and paths defined here. **This is the ONLY cell that changes between T5-small and SciFive runs.**

See SciFive Switch Guide (Section 8) for the 4 lines to change.

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME       = "t5-small"   # swap to SciFive: see Section 8
MODEL_SHORT_NAME = "t5_small"   # used in file paths

# ── Prefix ─────────────────────────────────────────────────────────────────
INPUT_PREFIX = "simplify: "

# ── Tokenizer limits ───────────────────────────────────────────────────────
MAX_INPUT_LENGTH  = 256
MAX_TARGET_LENGTH = 128

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE       = 8    # set to 4 for SciFive
GRAD_ACCUM_STEPS = 1    # set to 2 for SciFive (effective batch stays 8)
LEARNING_RATE    = 5e-4
NUM_EPOCHS       = 10
SEED             = 42

# ── Generation ─────────────────────────────────────────────────────────────
NUM_BEAMS = 4

# ── Paths ──────────────────────────────────────────────────────────────────
OUTPUT_DIR           = f"checkpoints/{MODEL_SHORT_NAME}"
DRIVE_CHECKPOINT_DIR = f"/drive/MyDrive/NLP_Project/checkpoints/{MODEL_SHORT_NAME}"
PREDICTIONS_FILE     = f"predictions/{MODEL_SHORT_NAME}.jsonl"

# ── Optional ───────────────────────────────────────────────────────────────
# Set WANDB_ENABLED=True and run !wandb login in a cell above config to enable W&B
WANDB_ENABLED = False

print(f"MODEL_NAME:        {MODEL_NAME}")
print(f"MODEL_SHORT_NAME:  {MODEL_SHORT_NAME}")
print(f"MAX_INPUT_LENGTH:  {MAX_INPUT_LENGTH}, MAX_TARGET_LENGTH: {MAX_TARGET_LENGTH}")
print(f"BATCH_SIZE:        {BATCH_SIZE}, GRAD_ACCUM_STEPS: {GRAD_ACCUM_STEPS}")
print(f"LEARNING_RATE:     {LEARNING_RATE}, NUM_EPOCHS: {NUM_EPOCHS}, SEED: {SEED}")
print(f"OUTPUT_DIR:        {OUTPUT_DIR}")
print(f"PREDICTIONS_FILE:  {PREDICTIONS_FILE}")

## 3. Imports & Seeds

In [ ]:
import random, json, os, sys, shutil
import numpy as np
import pandas as pd
import torch
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)
from datasets import load_from_disk

# After kernel restart, cwd resets to /content — navigate back to project root
_COLAB_PROJECT = '/content/NLP-Project'
if os.path.exists(_COLAB_PROJECT):
    os.chdir(_COLAB_PROJECT)
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.metrics import compute_sari, compute_bleu, compute_fkgl

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROCESSED_DIR   = os.path.join(project_root, 'data/processed')
PREDICTIONS_DIR = os.path.join(project_root, 'predictions')
RESULTS_DIR     = os.path.join(project_root, 'results')

print('Imports done.')
print('CUDA available:', torch.cuda.is_available())
print('Project root:', project_root)

## 4. Load Data

In [ ]:
dataset = load_from_disk(PROCESSED_DIR)
train_dataset = dataset['train']
val_dataset   = dataset['val']
test_dataset  = dataset['test']

print(f'Train: {len(train_dataset):,} examples')
print(f'Val:   {len(val_dataset):,} examples')
print(f'Test:  {len(test_dataset):,} examples')
assert len(test_dataset) == 1046, f'Expected 1046 test rows, got {len(test_dataset)}'
print('Test size assertion passed.')

## 5. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Store val source texts for SARI computation in compute_metrics
eval_sources = val_dataset['source']

def preprocess_function(examples):
    inputs = [INPUT_PREFIX + s for s in examples['source']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(
        text_target=examples['target'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

orig_cols = train_dataset.column_names
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=orig_cols)
tokenized_val   = val_dataset.map(preprocess_function,   batched=True, remove_columns=orig_cols)
tokenized_test  = test_dataset.map(preprocess_function,  batched=True, remove_columns=orig_cols)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=None, padding=True)

print(f'Tokenized train columns: {tokenized_train.column_names}')
print(f'Shapes — train: {tokenized_train.shape}, val: {tokenized_val.shape}')

## 6. compute_metrics Helper

Used by Seq2SeqTrainer to select the best checkpoint by validation SARI.

In [ ]:
# compute_metrics not used during training
# eval_loss used as checkpoint metric (avoids fp16 overflow in beam search)
# SARI computed post-training in inference cell
compute_metrics = None

## 7. Model & Training — US1: T5-Small

Run this section with `MODEL_NAME = "t5-small"` (default config).

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to('cuda')
# Update data_collator now that model is available
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {MODEL_NAME}')
print(f'Trainable parameters: {n_params:,}')

In [ ]:
training_args = Seq2SeqTrainingArguments(

    output_dir=DRIVE_CHECKPOINT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    per_device_eval_batch_size=4,

    learning_rate=LEARNING_RATE,

    eval_strategy='epoch',

    save_strategy='epoch',

    load_best_model_at_end=True,

    metric_for_best_model='eval_loss',

    greater_is_better=False,




    seed=SEED,

    fp16=True,

    report_to='wandb' if WANDB_ENABLED else 'none',

    logging_steps=50,

)

print('TrainingArguments set.')

In [ ]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_val,

    processing_class=tokenizer,

    data_collator=data_collator,


)

print('Trainer ready.')

In [ ]:
trainer.train(resume_from_checkpoint=True)



best_sari = trainer.state.best_metric

best_ckpt = trainer.state.best_model_checkpoint

print(f'\nTraining complete.')

print(f'Best validation SARI: {best_sari:.4f}')

print(f'Best checkpoint:      {best_ckpt}')

assert os.path.isdir(DRIVE_CHECKPOINT_DIR), f"Checkpoint dir missing: {DRIVE_CHECKPOINT_DIR}", f'Checkpoint dir missing: {OUTPUT_DIR}'

In [ ]:
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
shutil.copytree(OUTPUT_DIR, DRIVE_CHECKPOINT_DIR, dirs_exist_ok=True)
print(f'Checkpoint saved to Drive: {DRIVE_CHECKPOINT_DIR}')
drive_files = os.listdir(DRIVE_CHECKPOINT_DIR)
assert len(drive_files) > 0, 'Drive checkpoint directory is empty'
print(f'Drive checkpoint files: {drive_files[:5]}')

## 8. SciFive Switch Guide — US2

To train SciFive, change **exactly these 4 lines** in the Config cell (Section 2), then re-run all cells from Section 2 onward:

```python
MODEL_NAME       = "razent/SciFive-base-Pubmed_PMC"
MODEL_SHORT_NAME = "scifive"
BATCH_SIZE       = 4    # SciFive-base uses more VRAM than T5-small
GRAD_ACCUM_STEPS = 2    # effective batch remains 8
```

**No other cell changes.** Effective batch size = BATCH_SIZE × GRAD_ACCUM_STEPS = 8 for both models.

**If session times out** (SciFive ~10–12 hours): use `trainer.train(resume_from_checkpoint=True)` in the train cell.

**Validation**: After SciFive training, `checkpoints/scifive/` must exist with model weight files and be copied to Drive.

## 9. Inference & Metrics — US3

Run after training (US1 or US2). Loads best checkpoint, generates predictions on test set, saves JSONL and metrics.

In [ ]:
# Load best checkpoint — resolves checkpoint-NNNN subfolder automatically
import glob as _glob, json as _json

def _find_best_checkpoint(drive_dir):
    # 1. trainer still in scope from training cell
    try:
        ckpt = trainer.state.best_model_checkpoint
        if ckpt and os.path.isdir(ckpt):
            return ckpt
    except NameError:
        pass
    # 2 & 3. Parse trainer_state.json saved by Trainer after training
    state_file = os.path.join(drive_dir, 'trainer_state.json')
    if os.path.exists(state_file):
        state = _json.load(open(state_file))
        ckpt = state.get('best_model_checkpoint', '')
        if ckpt and os.path.isdir(ckpt):
            return ckpt
        # best_model_checkpoint path stale — derive from log_history
        history = state.get('log_history', [])
        eval_entries = [e for e in history if 'eval_loss' in e]
        if eval_entries:
            best_entry = min(eval_entries, key=lambda e: e['eval_loss'])
            ckpt = os.path.join(drive_dir, f'checkpoint-{int(best_entry["step"])}')
            if os.path.isdir(ckpt):
                return ckpt
    # 4. Last resort: highest-numbered checkpoint dir
    ckpts = sorted(
        _glob.glob(os.path.join(drive_dir, 'checkpoint-*')),
        key=lambda p: int(p.rsplit('-', 1)[-1])
    )
    return ckpts[-1] if ckpts else drive_dir

BEST_CHECKPOINT = _find_best_checkpoint(DRIVE_CHECKPOINT_DIR)
print(f'Loading from: {BEST_CHECKPOINT}')

infer_model = AutoModelForSeq2SeqLM.from_pretrained(BEST_CHECKPOINT).to('cuda')
infer_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
infer_model.eval()

INFER_BATCH = 16
rows = []

for batch_start in range(0, len(test_dataset), INFER_BATCH):
    batch   = test_dataset[batch_start:batch_start + INFER_BATCH]
    sources = batch['source']
    refs    = batch['target']

    inputs = infer_tokenizer(
        [INPUT_PREFIX + s for s in sources],
        return_tensors='pt', padding=True,
        max_length=MAX_INPUT_LENGTH, truncation=True,
    ).to('cuda')

    with torch.no_grad():
        out_ids = infer_model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_TARGET_LENGTH,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    preds = infer_tokenizer.batch_decode(out_ids, skip_special_tokens=True)
    for src, pred, ref in zip(sources, preds, refs):
        rows.append({'source': src, 'prediction': pred, 'reference': ref})

os.makedirs(PREDICTIONS_DIR, exist_ok=True)
with open(PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

assert len(rows) == 1046, f'Expected 1046 rows, got {len(rows)}'
print(f'Saved {PREDICTIONS_FILE} ({len(rows)} rows)')
print('Sample:', rows[0]['prediction'][:120])

In [ ]:
loaded      = [json.loads(l) for l in open(PREDICTIONS_FILE)]
sources_lst = [r['source']     for r in loaded]
preds_lst   = [r['prediction'] for r in loaded]
refs_lst    = [r['reference']  for r in loaded]

print('Computing SARI...')
sari = compute_sari(sources_lst, preds_lst, refs_lst)
print('Computing BLEU...')
bleu = compute_bleu(preds_lst, refs_lst)
print('Computing FKGL (predictions)...')
fkgl_output = compute_fkgl(preds_lst)
print('Computing FKGL (sources)...')
fkgl_input  = compute_fkgl(sources_lst)
fkgl_delta  = fkgl_output - fkgl_input

metrics_dict = {
    'system':      MODEL_SHORT_NAME,
    'sari':        round(sari,        4),
    'bleu':        round(bleu,        4),
    'fkgl_input':  round(fkgl_input,  4),
    'fkgl_output': round(fkgl_output, 4),
    'fkgl_delta':  round(fkgl_delta,  4),
}
print(f'\nResults for {MODEL_SHORT_NAME}:')
for k, v in metrics_dict.items():
    print(f'  {k:<12}: {v}')

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics_path = os.path.join(RESULTS_DIR, 'metrics.csv')

new_row = pd.DataFrame([metrics_dict])
if os.path.exists(metrics_path):
    existing = pd.read_csv(metrics_path)
    existing = existing[existing['system'] != MODEL_SHORT_NAME]
    combined = pd.concat([existing, new_row], ignore_index=True)
else:
    combined = new_row

combined.to_csv(metrics_path, index=False)
print(f'Saved {metrics_path}')
print(combined.to_string(index=False))

In [ ]:
df = pd.read_csv(os.path.join(RESULTS_DIR, 'metrics.csv'))
assert MODEL_SHORT_NAME in df['system'].values, f'Row missing for {MODEL_SHORT_NAME}'

row = df[df['system'] == MODEL_SHORT_NAME].iloc[0]
assert row['fkgl_delta'] < 0, f'FKGL must decrease (Principle III): {row["fkgl_delta"]}'
assert 0 <= row['sari'] <= 100
assert 0 <= row['bleu'] <= 100

print(f'\n✓ {MODEL_SHORT_NAME} assertions passed')
print(f'  SARI:       {row["sari"]:.4f}')
print(f'  BLEU:       {row["bleu"]:.4f}')
print(f'  FKGL delta: {row["fkgl_delta"]:.4f} (negative = simpler)')

## 10. W&B (Optional) — US4

W&B logging is controlled by `WANDB_ENABLED` in the Config cell.

**To enable**:
1. Run `!wandb login` in a new cell above Config
2. Set `WANDB_ENABLED = True` in Config
3. Re-run all cells from Config onward

`report_to='wandb' if WANDB_ENABLED else 'none'` in TrainingArguments handles the switch automatically.  
When `WANDB_ENABLED = False` (default), no `import wandb` is required and training runs without W&B dependency.

## 11. Contract Compliance

Run after both T5-small and SciFive training + inference are complete.

In [ ]:
print('=== Contract Compliance Check ===\n')

# Check both prediction files
for sys_name in ['t5_small', 'scifive']:
    pred_file = os.path.join(PREDICTIONS_DIR, f'{sys_name}.jsonl')
    if not os.path.exists(pred_file):
        print(f'⚠ SKIP: {pred_file} not yet generated')
        continue
    pred_rows = [json.loads(l) for l in open(pred_file)]
    assert len(pred_rows) == 1046, f'{sys_name}: expected 1046 rows, got {len(pred_rows)}'
    for i, r in enumerate(pred_rows):
        assert r.get('source') and r.get('prediction') and r.get('reference'), f'Row {i} missing fields'
    print(f'✓ predictions/{sys_name}.jsonl — 1046 rows, all fields present')

# Check metrics.csv
metrics_path = os.path.join(RESULTS_DIR, 'metrics.csv')
assert os.path.exists(metrics_path), 'results/metrics.csv missing'
df_check = pd.read_csv(metrics_path)

for sys_name in ['t5_small', 'scifive']:
    if sys_name not in df_check['system'].values:
        print(f'⚠ SKIP: metrics.csv missing row for {sys_name}')
        continue
    r = df_check[df_check['system'] == sys_name].iloc[0]
    assert r['fkgl_delta'] < 0, f'{sys_name}: fkgl_delta must be negative'
    print(f'✓ metrics.csv — {sys_name} row present, fkgl_delta={r["fkgl_delta"]:.4f}')

if 'rule_based' in df_check['system'].values:
    rb = df_check[df_check['system'] == 'rule_based'].iloc[0]
    assert rb['fkgl_delta'] < 0
    print(f'✓ metrics.csv — rule_based row preserved, fkgl_delta={rb["fkgl_delta"]:.4f}')
else:
    print('⚠ WARNING: rule_based row missing from metrics.csv')

print('\n=== All compliance assertions passed ===')